In [1]:
###-----------------
### Import Libraries
###-----------------

from pathlib import Path  # Import Path for file system path operations and management
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split  # for train-test splitting
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score,ConfusionMatrixDisplay


import torch
import torch.nn as nn
from torchsummary import summary
from collections.abc import Callable    # type hinting callable objects/functions
from typing import Literal      # literal type hints to restrict variable values

from torch.utils.data import Dataset, DataLoader




In [2]:
###----------------------
### Some basic parameters
###----------------------

# inpDir = Path('..') / '..' / 'input'
# outDir = Path('..') / 'output'
# modelDir = Path('..') / 'models'
# subDir = 'fifa_2019'

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE)
# rng = np.random.default_rng(seed = RANDOM_STATE) # Set Random Seed for reproducible  results

EPOCHS = 75 # number of epochs


BATCH_SIZE = 64              # <------- Look at TRAIN_SIZE too
ALPHA = 0.001 # learning rate
TEST_SIZE = 0.2
NUM_GENERATE= 1000
TEMPERATURE=0.8
BUFFER_SIZE= 10000
SEQ_LENGTH=100
EMBEDDING_DIM=256
RNN_UNITS=1024
PATIENCE=10
MIN_DELTA=1E-4
LR_PATIENCE=5
LR_FACTOR=0.1


# parameters for Matplotlib
params = {'legend.fontsize': 'medium',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'medium',
          'axes.titlesize':'x-large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }

plt.rcParams.update(params)

CMAP = plt.cm.coolwarm
plt.style.use('seaborn-v0_8-darkgrid') # plt.style.use('ggplot')

In [3]:
# # Check if all directories are present
# outDir.mkdir(parents=True, exist_ok=True)

# modelSubDir = modelDir/ subDir
# modelSubDir.mkdir(parents=True, exist_ok=True)

In [4]:
filePath='shakespeare.txt'
text=open(filePath,'rb').read().decode(encoding='utf-8')
len(text)

1115395

In [5]:
print(text[:400])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it 


In [6]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed=RANDOM_STATE)
    device=torch.device('cuda')
else:
    device=torch.device('cpu')


In [7]:
vocab=sorted(set(text))
vocab_size=len(vocab)
vocab_size

65

In [8]:
char2idx={v:i for i, v in enumerate(vocab)}
idx2char=np.array(vocab)
print(char2idx)
text_as_int=np.array([char2idx[c] for c in text])
text_as_int

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


array([18, 47, 56, ...,  8,  0,  0], shape=(1115395,))

### dataset

In [9]:
#every time we request it will give batch of random 32 rows from dataset
class ShakespeareDataset(Dataset):

    def __init__(self,
                 text_as_int,
                seq_length):
        super(ShakespeareDataset,self).__init__()
        self.sequences=[]
        self.seq_length=seq_length
        self.text_as_int=text_as_int

        for i in range(0,len(text_as_int)-seq_length,seq_length):
            self.sequences.append(text_as_int[i:i+seq_length+1])
        self.sequences=np.array(self.sequences)

        # self.features=self.df.drop(label_col,axis=1).to_numpy()
        # self.features=se;f.geatures / 255
        # y=self.df[label_col].to_numpy()
        
        # self.X= torch.tensor(X, dtype=torch.float32)
        # self.y= torch.tensor(y, dtype=torch.long)

        # # if self.is train_df.head()train:
        # #     self.transformer=DataTransformer()
        # #     self.features, self.labels=self.transformer.fit_transform(X,y)
        # # else:
        # #     if self.transformer is None:
        # #         raise

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self,index):
        seq=self.sequences[index]
        input_seq=torch.tensor(seq[:-1],dtype=torch.long)
        target_seq=torch.tensor(seq[1:],dtype=torch.long)
        return input_seq, target_seq

In [10]:
dataset=ShakespeareDataset(text_as_int=text_as_int,seq_length=SEQ_LENGTH)
len(dataset)

11153

In [11]:
train_size=int((0.8*len(dataset))// BATCH_SIZE*BATCH_SIZE)
test_size=len(dataset)-train_size
train_dataset,test_dataset=torch.utils.data.random_split(
    dataset,[train_size,test_size],
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)

In [12]:
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)
len(train_loader),len(test_loader)

(139, 36)

In [13]:
inputs,targets=next(iter(train_loader))
first_input=inputs[0]
first_target=targets[0]
first_input,first_target

(tensor([50,  6,  0, 13, 50, 50,  1, 41, 47, 56, 41, 59, 51, 57, 58, 39, 52, 41,
         43, 57,  1, 61, 43, 50, 50,  1, 41, 53, 52, 57, 47, 42, 43, 56, 43, 42,
          8,  0, 37, 53, 59,  1, 57, 39, 63,  1, 58, 46, 39, 58,  1, 17, 42, 61,
         39, 56, 42,  1, 47, 57,  1, 63, 53, 59, 56,  1, 40, 56, 53, 58, 46, 43,
         56,  5, 57,  1, 57, 53, 52, 10,  0, 31, 53,  1, 57, 39, 63,  1, 61, 43,
          1, 58, 53, 53,  6,  1, 40, 59, 58,  1]),
 tensor([ 6,  0, 13, 50, 50,  1, 41, 47, 56, 41, 59, 51, 57, 58, 39, 52, 41, 43,
         57,  1, 61, 43, 50, 50,  1, 41, 53, 52, 57, 47, 42, 43, 56, 43, 42,  8,
          0, 37, 53, 59,  1, 57, 39, 63,  1, 58, 46, 39, 58,  1, 17, 42, 61, 39,
         56, 42,  1, 47, 57,  1, 63, 53, 59, 56,  1, 40, 56, 53, 58, 46, 43, 56,
          5, 57,  1, 57, 53, 52, 10,  0, 31, 53,  1, 57, 39, 63,  1, 61, 43,  1,
         58, 53, 53,  6,  1, 40, 59, 58,  1, 52]))

In [14]:
inp_chars=''.join([idx2char[idx] for idx in first_input.numpy()])
traget_chars=''.join([idx2char[idx] for idx in first_target.numpy()])
inp_chars,traget_chars

("l,\nAll circumstances well considered.\nYou say that Edward is your brother's son:\nSo say we too, but ",
 ",\nAll circumstances well considered.\nYou say that Edward is your brother's son:\nSo say we too, but n")

## define Model

In [15]:
class CharRNN(nn.Module):

    '''Layers:
    33  -> 16 -> 8-> 4
    
    '''
    def __init__(self,
                vocab_size,
                embedding_dim,
                hidden_units,
                num_layers=2,
                dropout=0.2):
        
        super(CharRNN,self).__init__()
        self.vocab_size=vocab_size
        self.hidden_units=hidden_units
        self.num_layers=num_layers

        #Embedding Layer
        self.embedding=nn.Embedding(vocab_size,embedding_dim)

        #GRU layer
        self.gru=nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_units,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers>1 else 0,
            bidirectional=False
        )

        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(hidden_units,vocab_size)
        
    def forward(self,x, hidden=None):
        if hidden is None:
            hidden=self.init_hidden(x.size(0),x.device)
        embedded=self.embedding(x)
        output,hidden=self.gru(embedded,hidden)
        output=self.dropout(output)
        output=output.reshape(-1,self.hidden_units)
        logits=self.fc(output)
        logits=logits.view(x.size(0),-1,self.vocab_size)
        return logits,hidden

    def init_hidden(self,batch_size,device):
        return torch.zeros(self.num_layers,batch_size,self.hidden_units,device=device)

input_dim=34
model=CharRNN(vocab_size=vocab_size,
                embedding_dim=EMBEDDING_DIM,
                hidden_units=RNN_UNITS,
                num_layers=2,
                dropout=0.2).to(device=device)

model
        
       

CharRNN(
  (embedding): Embedding(65, 256)
  (gru): GRU(256, 1024, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=1024, out_features=65, bias=True)
)

In [16]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=ALPHA)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,
                                                    mode='min',
                                                    factor=LR_FACTOR,
                                                    patience=LR_PATIENCE)

## Train on one Epoch

In [17]:
def train_model(model,
                train_loader,
                loss_fn=loss_fn,
                optimizer=optimizer,
                scheduler=scheduler,
                device=device):
    
    model.train()

    total_loss = 0

    for train_X, train_y in train_loader:
        train_X,train_y = train_X.to(device), train_y.to(device)
        hidden=model.init_hidden(train_X.size(0),device) #hidden state (a_t)
        optimizer.zero_grad()
        logits,hidden=model(train_X,hidden) #forward pass
        batch_loss = loss_fn(logits.reshape(-1,vocab_size),train_y.reshape(-1))

        batch_loss.backward()

        #Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.0)
        optimizer.step()
        total_loss+=batch_loss

        hidden=hidden.detach()

    avg_loss=total_loss/len(train_loader)
    return avg_loss
        

In [18]:
def evaluate(model,test_loader,loss_fn=loss_fn, device=device):
    model.eval()  #stop learning

    total_loss = 0

    with torch.inference_mode():
        for test_X, test_y in test_loader:
            test_X,test_y = test_X.to(device), test_y.to(device)
            hidden=model.init_hidden(test_X.size(0),device) #hidden state (a_t)
            logits,hidden=model(test_X,hidden) 
            batch_loss = loss_fn(logits.reshape(-1,vocab_size),test_y.reshape(-1))
            total_loss+=batch_loss.item()

    avg_loss=total_loss/len(test_loader)
    return avg_loss

In [ ]:
history={'epoch':[],
        'train_loss':[],
        'test_loss':[],
        'train_rmse':[],
        'test_rmse':[]
        }
best_test_loss=float('inf')
best_model_state=None

for epoch in range(EPOCHS):
    #Train
    train_loss=train_model(model,train_loader)
    train_rmse=np.sqrt(train_loss.detach().cpu().item())
    
    #Test
    test_loss=evaluate(model,test_loader)
    test_rmse=np.sqrt(test_loss)

    scheduler.step(test_loss)
    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['train_rmse'].append(train_rmse)
    history['test_rmse'].append(test_rmse)  

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_state = model.state_dict().copy()

    if epoch % 10 == 0:
        # fmtstr= | Loss:{:.5f} / {:.5f} | Acc: {:.5f} / {:.5f}
        print(f'Epoch:{epoch:5d}/{EPOCHS:5d} |\
              LR:{optimizer.param_groups[0] ['lr']:0.6f} | \
              Train Loss:{train_loss:.4f} |\
              Test Loss:{test_loss:.4f}')

print(f'Best loss:{best_test_loss}')
if best_model_state is not None:
    model.load_state_dict(best_model_state)


Epoch:    0/   75 |              LR:0.001000 |               Train Loss:1.3647 |              Test Loss:1.3581
Epoch:   10/   75 |              LR:0.000100 |               Train Loss:0.9020 |              Test Loss:1.3852
Epoch:   20/   75 |              LR:0.000010 |               Train Loss:0.7320 |              Test Loss:1.4717
Epoch:   30/   75 |              LR:0.000000 |               Train Loss:0.7263 |              Test Loss:1.4747


In [ ]:
loss_df=pd.DataFrame(history)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
import matplotlib. pyplot as plt
from sklearn.metrics import  confusion_matrix, ConfusionMatrixDisplay, f1_score
from scipy.interpolate import make_interp_spline

def fn_plot_torch_hist(hist_df: pd.DataFrame):
    """
    Plots the training and validation loss and accuracy curves from a PyTorch training history DataFrame.
    
    Args:
        hist_df (pd.DataFrame): A pandas DataFrame with five columns:
                                - First column: epoch (x-axis values)
                                - Second & third columns: losses (train & validation)
                                - Fourth & fifth columns: accuracies (train & validation)
    
    Returns:
        None: Displays the matplotlib plots.
    """
    # Ensure the DataFrame has exactly five columns
    if hist_df.shape[1] < 5:
        raise ValueError("The DataFrame must have atleast five columns: epoch, train_loss, val_loss, train_acc, val_acc.")
    
    # Extract column names for better readability and maintainability
    #epoch loss	           test_loss	 acc	        test_acc
    x_col, train_loss_col, val_loss_col, train_acc_col, val_acc_col, *cols= hist_df.columns
    
    # Instantiate figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    facecolor = 'cyan'  # Text box background color
    fontsize = 12  # Font size for annotations
    props = dict(boxstyle='round', facecolor=facecolor, alpha=0.5)  # Text box properties

    # First subplot: Loss curves
    ax = axes[0]
    hist_df.plot(x=x_col, y=[train_loss_col, val_loss_col], ax=ax)
    
    # Annotation: Final losses
    final_losses_text = f"Loss:\nTrain: {hist_df[train_loss_col].iloc[-1]:.4f}\nVal: {hist_df[val_loss_col].iloc[-1]:.4f}"
    ax.text(0.3, 0.95, final_losses_text, transform=ax.transAxes, fontsize=fontsize, 
            verticalalignment='top', bbox=props)
    
    # Annotation: Minimum validation loss
    best_val_loss_idx = hist_df[val_loss_col].idxmin()
    best_val_loss_epoch = hist_df.loc[best_val_loss_idx, x_col]
    best_val_loss_value = hist_df.loc[best_val_loss_idx, val_loss_col]
    ax.annotate(f"Min: {best_val_loss_value:.4f}", 
                xy=(best_val_loss_epoch, best_val_loss_value), 
                xytext=(best_val_loss_epoch - 2, best_val_loss_value + 0.05),
                fontsize=fontsize, ha='right', va='bottom', bbox=props, 
                arrowprops=dict(facecolor=facecolor, shrink=0.05))
    ax.axvline(x=best_val_loss_epoch, color='green', linestyle='-.', lw=2)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Loss Curves")
    ax.legend(loc='upper left')
    ax.grid(True)

    # Second subplot: Accuracy curves
    ax = axes[1]
    hist_df.plot(x=x_col, y=[train_acc_col, val_acc_col], ax=ax)
    
    # Annotation: Final accuracies
    final_acc_text = f"Accuracy:\nTrain: {hist_df[train_acc_col].iloc[-1]:.4f}\nVal: {hist_df[val_acc_col].iloc[-1]:.4f}"
    ax.text(0.3, 0.2, final_acc_text, transform=ax.transAxes, fontsize=fontsize, 
            verticalalignment='top', bbox=props)
    
    # Annotation: corresponding validation accuracy
    best_val_acc_epoch = hist_df.loc[best_val_loss_idx, x_col]
    best_val_acc_value = hist_df.loc[best_val_loss_idx, val_acc_col]
    ax.annotate(f"Max: {best_val_acc_value:.4f}", 
                xy=(best_val_acc_epoch, best_val_acc_value), 
                xytext=(best_val_acc_epoch - 2, best_val_acc_value - 0.05),
                fontsize=fontsize, ha='right', va='top', bbox=props, 
                arrowprops=dict(facecolor=facecolor, shrink=0.05))
    ax.axvline(x=best_val_acc_epoch, color='green', linestyle='-.', lw=2)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Accuracy Curves")
    ax.legend(loc='lower left')
    ax.grid(True)

    plt.tight_layout()
    plt.show()


def fn_plot_confusion_matrix(y_true, y_pred, labels):
    '''
    Args:
        y_true: Ground Truth 
        y_pred : Predictions
        labels : dictionary 
                  {0: 'Goal Keeper', 
                  1: 'Defender', 
                  2: 'Mid-Fielder', 
                  3: 'Forward'}
    
    '''
    
    cm  = confusion_matrix(y_true, y_pred)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=labels.values())
    
    fig, ax = plt.subplots(figsize = (4,4))
    
    disp.plot(ax = ax, cmap = 'Blues', xticks_rotation = 'vertical', colorbar=False)

    # Disable the grid
    ax.grid(False)

    title_str = f'F1 Score : {f1_score(y_true, y_pred, average='weighted'):0.5f}'
    ax.set_title(title_str)

    plt.show()

In [ ]:
fn_plot_torch_hist(loss_df)

In [ ]:
@torch.inference_mode()
def generate_text(model,start_string,num_generate=NUM_GENERATE,temperature=TEMPERATURE)
    model.eval()
    input_indices=[char2idx.get(ch,0) for ch in start_string]
    print(f'Start String:{start_string}| Indices:{input_indices}')
    input_tensor=torch.tensor([input_indices],dtype=torch.long.device=device)
    hidden=model.init_hidden(1,device)
    generated_chars=[]
    for _ in range(num_generate):
        logits,hidden=model(input_tensor,hidden)
        logits=logits[0,-1,:]/temperature
        prob=torch.softmax(logits,dim=-1)
        next_char_idx=torch.multinomial(probs,num_samples=1).item()
        generated_chars.append(idx2char[next_char_idx])
        input_tensor=torch.tensor([[next_char_idx]], dtype=torch.long,device=device)
        hidden=hidden.detach()
    return start_string+' '.join(generated_chars)

In [ ]:
start_string='First'

In [ ]:
data_df = pd.read_csv('./fashion-mnist_train.csv',header=0)
data_df.shape

In [ ]:
#class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
#               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

class_names = {0: 'T-shirt/top',1:'Trouser',2:'Pullover',3:'Dress',4:'Coat',
               5:'Sandal', 6: 'Shirt',7: 'Sneaker', 8:'Bag', 9: 'Ankle boot'}

In [ ]:
X_train = data_df.drop('label', axis = 1).to_numpy()
y_train = data_df['label'].to_numpy()

X_test = test_df.drop('label', axis = 1).to_numpy()
y_test = test_df['label'].to_numpy()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# splitting in train ans test datasets
X_train,X_test,y_train,y_test = train_test_split(features,labels,
                                     stratify=labels,
                                     train_size=TRAIN_SIZE, 
                                     random_state=RANDOM_STATE )
X_train.shape,X_test.shape,y_train.shape,y_test.shape

In [ ]:
list(class_names.keys())

In [ ]:
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
Label_col=34
train_dsIonoModel(train_df,batch_size=True,label_col=label_col)
test_ds=IonoModel(test_df,is-train=False,label_col=label_col,transformer=train_ds.transformer)


In [ ]:
train_loader=DataLoader(train_ds,
                       batch_size=True,
                       shuffle=True)


In [ ]:
train_dataset=FifaDataset(X_train,y_train)

#dataloader shuffles the data
train_loader=DataLoader(dataset=train_dataset,batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
#ways to select batch
# 1.complete dataset- everytime u take unique 32 rows,
# 2. everytime u select at random but from same lot, rows can be repeated
# *gpu works very well with 2 raise to power something, so select rows in 2^
# #last batch should be full batch size as previous batches, 
#less rows in last batch may create problem, they may give more importance
for batch_idx, (data,target) in enumerate(train_loader):
    print(f'Batch: {batch_idx +1}: ', end=' ')
    print(f'data: {data.shape}: ', end=' ')
    print(f'Target: {target.shape}: ')
    

In [ ]:
X_train.shape, X_test.shape

In [ ]:
test_dataset=FifaDataset(X_test,y_test.to_numpy())
test_loader=DataLoader(dataset=test_dataset,batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
for batch_idx, (data,target) in enumerate(test_loader):
    print(f'Batch: {batch_idx +1}: ', end=' ')
    print(f'data: {data.shape}: ', end=' ')
    print(f'Target: {target.shape}: ')
    

# Defne model

In [ ]:
#in sequnettial class, all layers given in sequence, one by one layer,
# but if we want to jump from layer 1 to layer 3 it is not possibl
# , for use below and other methods also available


class FifaModel(nn.Module):

    '''Layers:
    33  -> 16 -> 8-> 4
    
    '''
    def __init__(self,input_dim):
        super(FifaModel,self).__init__()
        self.layer1=nn.Linear(input_dim,16)
        self.active1=nn.ReLU()
        
        self.layer2=nn.Linear(16,8)
        self.active2=nn.ReLU()

        self.layer3=nn.Linear(8,4)

    def forward(self,x):
        x=self.layer1(x)
        x=self.active1(x)
        x=self.layer2(x)
        x=self.active2(x)
        x=self.layer3(x)

        return x

    
        
        

In [ ]:
model=FifaModel(input_dim=X_train.shape[1]).to(device=device)


In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=ALPHA)

loss, tloss, n_epoch, acc, tacc=[],[],[],[],[]

for epoch in range(EPOCHS):
    model.train()

    epoch_loss = 0
    epoch_acc = 0
    tepoch_loss = 0
    tepoch_acc = 0

    for batch_idx,(train_X, train_y) in enumerate(train_loader):
        train_X,train_y = train_X.to(device), train_y.to(device)
        predict_prob = model(train_X)
        batch_loss = loss_fn(predict_prob,train_y)
        epoch_loss += (batch_loss-epoch_loss)/(batch_idx+1)

# '''epoch loss started with 0, batch loss is calculated,
#         in first loop-epoch loss= batch loss
# epoch-loss = average of batch loss in one complete paas(epcoh)
#        '''

        #backward propagation
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()


        #collecting the predictions and calculating accuracy
        _, y_pred = torch.max(predict_prob,1)
        batch_acc= accuracy_score(train_y.cpu().numpy(),y_pred.data.cpu())

        epoch_acc += (batch_acc -epoch_acc)/(batch_idx +1 )

    loss.append(epoch_loss.data.item())
    acc.append(epoch_acc)

    
    model.eval()

    with torch.inference_mode():
        for batch_idx, (test_X,test_y) in enumerate(test_loader):
            test_X,test_y=test_X.to(device) ,test_y.to(device)
            predict_prob_tst=model(test_X)
            tbatch_loss= loss_fn(predict_prob_tst,test_y)
            tepoch_loss += (tbatch_loss -tepoch_loss)/ (batch_idx+1)

             #collecting the predictions and calculating accuracy
            _, y_pred = torch.max(predict_prob_tst,1)
            tbatch_acc= accuracy_score(test_y.cpu().numpy(),y_pred.data.cpu())
            tepoch_acc += (tbatch_acc -tepoch_acc)/(batch_idx +1 )
        tloss.append(tepoch_loss.data.item())
        tacc.append(tepoch_acc)
    n_epoch.append(epoch)

    if epoch % 20==0:
        fmtStr= 'Epoch:{:5d}/{:5d} | Loss: {:.5f}/{:.5f} | Accuracy: {:.5f}/{:.5f}'
        print(fmtStr.format(epoch, EPOCHS,
                            epoch_loss.data.item(), 
                            tepoch_loss.data.item(),
                           epoch_acc,tepoch_acc))
        

In [ ]:
loss_df= pd.DataFrame({'epoch': n_epoch,
                      'loss':loss,
                      'test_loss':tloss,
                     'acc':acc,
                      'test_acc': tacc})

loss_df.head()

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
import matplotlib. pyplot as plt
from sklearn.metrics import  confusion_matrix, ConfusionMatrixDisplay, f1_score
from scipy.interpolate import make_interp_spline

def fn_plot_torch_hist(hist_df: pd.DataFrame):
    """
    Plots the training and validation loss and accuracy curves from a PyTorch training history DataFrame.
    
    Args:
        hist_df (pd.DataFrame): A pandas DataFrame with five columns:
                                - First column: epoch (x-axis values)
                                - Second & third columns: losses (train & validation)
                                - Fourth & fifth columns: accuracies (train & validation)
    
    Returns:
        None: Displays the matplotlib plots.
    """
    # Ensure the DataFrame has exactly five columns
    if hist_df.shape[1] < 5:
        raise ValueError("The DataFrame must have atleast five columns: epoch, train_loss, val_loss, train_acc, val_acc.")
    
    # Extract column names for better readability and maintainability
    #epoch loss	           test_loss	 acc	        test_acc
    x_col, train_loss_col, val_loss_col, train_acc_col, val_acc_col, *cols= hist_df.columns
    
    # Instantiate figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    facecolor = 'cyan'  # Text box background color
    fontsize = 12  # Font size for annotations
    props = dict(boxstyle='round', facecolor=facecolor, alpha=0.5)  # Text box properties

    # First subplot: Loss curves
    ax = axes[0]
    hist_df.plot(x=x_col, y=[train_loss_col, val_loss_col], ax=ax)
    
    # Annotation: Final losses
    final_losses_text = f"Loss:\nTrain: {hist_df[train_loss_col].iloc[-1]:.4f}\nVal: {hist_df[val_loss_col].iloc[-1]:.4f}"
    ax.text(0.3, 0.95, final_losses_text, transform=ax.transAxes, fontsize=fontsize, 
            verticalalignment='top', bbox=props)
    
    # Annotation: Minimum validation loss
    best_val_loss_idx = hist_df[val_loss_col].idxmin()
    best_val_loss_epoch = hist_df.loc[best_val_loss_idx, x_col]
    best_val_loss_value = hist_df.loc[best_val_loss_idx, val_loss_col]
    ax.annotate(f"Min: {best_val_loss_value:.4f}", 
                xy=(best_val_loss_epoch, best_val_loss_value), 
                xytext=(best_val_loss_epoch - 2, best_val_loss_value + 0.05),
                fontsize=fontsize, ha='right', va='bottom', bbox=props, 
                arrowprops=dict(facecolor=facecolor, shrink=0.05))
    ax.axvline(x=best_val_loss_epoch, color='green', linestyle='-.', lw=2)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Loss Curves")
    ax.legend(loc='upper left')
    ax.grid(True)

    # Second subplot: Accuracy curves
    ax = axes[1]
    hist_df.plot(x=x_col, y=[train_acc_col, val_acc_col], ax=ax)
    
    # Annotation: Final accuracies
    final_acc_text = f"Accuracy:\nTrain: {hist_df[train_acc_col].iloc[-1]:.4f}\nVal: {hist_df[val_acc_col].iloc[-1]:.4f}"
    ax.text(0.3, 0.2, final_acc_text, transform=ax.transAxes, fontsize=fontsize, 
            verticalalignment='top', bbox=props)
    
    # Annotation: corresponding validation accuracy
    best_val_acc_epoch = hist_df.loc[best_val_loss_idx, x_col]
    best_val_acc_value = hist_df.loc[best_val_loss_idx, val_acc_col]
    ax.annotate(f"Max: {best_val_acc_value:.4f}", 
                xy=(best_val_acc_epoch, best_val_acc_value), 
                xytext=(best_val_acc_epoch - 2, best_val_acc_value - 0.05),
                fontsize=fontsize, ha='right', va='top', bbox=props, 
                arrowprops=dict(facecolor=facecolor, shrink=0.05))
    ax.axvline(x=best_val_acc_epoch, color='green', linestyle='-.', lw=2)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Accuracy Curves")
    ax.legend(loc='lower left')
    ax.grid(True)

    plt.tight_layout()
    plt.show()


def fn_plot_confusion_matrix(y_true, y_pred, labels):
    '''
    Args:
        y_true: Ground Truth 
        y_pred : Predictions
        labels : dictionary 
                  {0: 'Goal Keeper', 
                  1: 'Defender', 
                  2: 'Mid-Fielder', 
                  3: 'Forward'}
    
    '''
    
    cm  = confusion_matrix(y_true, y_pred)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=labels.values())
    
    fig, ax = plt.subplots(figsize = (4,4))
    
    disp.plot(ax = ax, cmap = 'Blues', xticks_rotation = 'vertical', colorbar=False)

    # Disable the grid
    ax.grid(False)

    title_str = f'F1 Score : {f1_score(y_true, y_pred, average='weighted'):0.5f}'
    ax.set_title(title_str)

    plt.show()

In [ ]:
fn_plot_torch_hist(loss_df)

In [ ]:
y_true,y_pred=[],[]

model.eval()
with torch.inference_mode():

    for batch_idx,(train_X, train_y) in enumerate(train_loader):
        train_X,train_y = train_X.to(device), train_y.to(device)
        predict= model(train_X)
        y_pred.extend(torch.argmax(pred,dim=1).cpu().numpy())
        y_true.extend(train_y)

    y_true=np.concatenate(y_true)
        